# LM-Prior Cold-Start Recommender System (Google Colab Version)
**Phase 3: Speed Optimized**
This notebook has been optimized with batched GPU evaluation, PyTorch GPU distance calculations, and fully vectorized prior calculations.


In [ ]:
!pip install -q sentence-transformers google-genai tqdm scipy scikit-learn pandas numpy matplotlib


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
import time
import sys
from tqdm import tqdm
from scipy.spatial import distance
import torch.nn as nn
from multiprocessing import Process, Queue

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
DATASET = 'amazon' # 'movielens' | 'bookcrossing' | 'amazon'
LM_PRIOR = 'bge' # 'sbert' | 'mpnet' | 'gemini' | 'bge'
GEMINI_API_KEY = 'YOUR_KEY_HERE'
AMAZON_TEXT_COL = 'reviewText' # 'summary' | 'reviewText'

NUM_EPOCHS = 20
RHO = 0.001
LR = 0.001
BATCH_SIZE = 64
HIDDEN_UNITS = 64
K = 71

MODEL_ID_MAP = {
    'sbert': 'all-MiniLM-L6-v2',
    'mpnet': 'all-mpnet-base-v2',
    'bge': 'BAAI/bge-large-en-v1.5'
}
RUN_NAME = f'{DATASET}_{LM_PRIOR}'

from google.colab import drive
drive.mount('/content/drive')

WORKING_DIR = '/content/drive/MyDrive/AmazonLM'
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)
os.makedirs('./model_save', exist_ok=True)
os.makedirs('./results', exist_ok=True)
print(f'Working directory: {os.getcwd()}')



## Core Utilities (Inlined from utils.py)


In [ ]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0


import sys
import copy
import torch
import random
import numpy as np
from collections import defaultdict
from multiprocessing import Process, Queue

# sampler for batch generation
def random_neq(l, r, s):
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t


def sample_function(user_train, item_emb, item_llm, usernum, itemnum, item_emb_dim, batch_size, maxlen, train, train_rate, result_queue, SEED):
    def sample():
        if train:
            user = np.random.randint(1, int(usernum*train_rate) + 1)
            while user not in user_train or len(user_train[user]) <= 1: 
                user = np.random.randint(1, int(usernum*train_rate) + 1)
        else:
            user = np.random.randint(int(usernum*train_rate) + 1, usernum + 1)
            while user not in user_train or len(user_train[user]) <= 1: 
                user = np.random.randint(int(usernum*train_rate) + 1, usernum + 1)
        
        # Items
        seq = np.zeros([maxlen], dtype=np.int32)
        pos = np.zeros([maxlen], dtype=np.int32)
        nxt = user_train[user][-1]
        idx = maxlen - 1

        ts = set(user_train[user])
        for i in reversed(user_train[user][:-1]):
            seq[idx] = i
            pos[idx] = nxt
            nxt = i
            idx -= 1
            if idx == -1: break
        
        # Item embedding
        seq_emb = [np.zeros(item_emb_dim)] * maxlen
        pos_emb = [np.zeros(item_emb_dim)] * maxlen
        neg_emb = [np.zeros(item_emb_dim)] * maxlen
        nxt_emb = item_emb[user][-1]
        idx = maxlen - 1
        
        for i_emb in reversed(item_emb[user][:-1]):
            seq_emb[idx] = i_emb
            pos_emb[idx] = nxt_emb
            
            # --- FIX: Sample real negative items from the pool ---
            if np.any(nxt_emb != 0): 
                neg_id = np.random.randint(1, itemnum + 1)
                while neg_id in ts:
                    neg_id = np.random.randint(1, itemnum + 1)
                neg_emb[idx] = item_llm[neg_id]
            
            nxt_emb = i_emb
            idx -= 1
            if idx == -1: break
        
        seq_emb = np.vstack(seq_emb)
        pos_emb = np.vstack(pos_emb)
        neg_emb = np.vstack(neg_emb)
        
        return (user, seq, pos, seq_emb, pos_emb, neg_emb)

    np.random.seed(SEED)
    while True:
        one_batch = []
        for i in range(batch_size):
            one_batch.append(sample())

        result_queue.put(zip(*one_batch))


class WarpSampler(object):
    def __init__(self, User, item_emb, item_llm, usernum, itemnum, item_emb_dim, train, train_rate, batch_size=64, maxlen=10, n_workers=1):
        self.result_queue = Queue(maxsize=n_workers * 10)
        self.processors = []
        for i in range(n_workers):
            self.processors.append(
                Process(target=sample_function, args=(User,
                                                      item_emb,
                                                      item_llm,
                                                      usernum,
                                                      itemnum,
                                                      item_emb_dim, 
                                                      batch_size,
                                                      maxlen,
                                                      train,
                                                      train_rate,
                                                      self.result_queue,
                                                      np.random.randint(2e9)
                                                      )))
            self.processors[-1].daemon = True
            self.processors[-1].start()

    def next_batch(self):
        return self.result_queue.get()

    def close(self):
        for p in self.processors:
            p.terminate()
            p.join()


# train/val/test data generation
def data_partition(fname):
    usernum = 0
    itemnum = 0
    User = defaultdict(list)
    user_train = {}
    user_valid = {}
    user_test = {}
    # assume user/item index starting from 1
    f = open('data/%s.txt' % fname, 'r')
    for line in f:
        u, i = line.rstrip().split(' ')
        u = int(u)
        i = int(i)
        usernum = max(u, usernum)
        itemnum = max(i, itemnum)
        User[u].append(i)

    for user in User:
        nfeedback = len(User[user])
        if nfeedback < 3:
            user_train[user] = User[user]
            user_valid[user] = []
            user_test[user] = []
        else:
            user_train[user] = User[user][:-2]
            user_valid[user] = []
            user_valid[user].append(User[user][-2])
            user_test[user] = []
            user_test[user].append(User[user][-1])
    return [user_train, user_valid, user_test, usernum, itemnum]

# TODO: merge evaluate functions for test and val set
# evaluate on test set
def evaluate(model, dataset, args):
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0

    if usernum>10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)
    for u in users:

        if len(train[u]) < 1 or len(test[u]) < 1: continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        seq[idx] = valid[u][0]
        idx -= 1
        for i in reversed(train[u]):
            seq[idx] = i
            idx -= 1
            if idx == -1: break
        rated = set(train[u])
        rated.add(0)
        item_idx = [test[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated: t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)

        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0] # - for 1st argsort DESC

        rank = predictions.argsort().argsort()[0].item()

        valid_user += 1

        if rank < 10:
            NDCG += 1 / np.log2(rank + 2)
            HT += 1
        if valid_user % 100 == 0:
            print('.', end="")
            sys.stdout.flush()

    return NDCG / valid_user, HT / valid_user


# evaluate on val set
def evaluate_valid(model, dataset, args):
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    valid_user = 0.0
    HT = 0.0
    if usernum>10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)
    for u in users:
        if len(train[u]) < 1 or len(valid[u]) < 1: continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        for i in reversed(train[u]):
            seq[idx] = i
            idx -= 1
            if idx == -1: break

        rated = set(train[u])
        rated.add(0)
        item_idx = [valid[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated: t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)

        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]

        rank = predictions.argsort().argsort()[0].item()

        valid_user += 1

        if rank < 10:
            NDCG += 1 / np.log2(rank + 2)
            HT += 1
        if valid_user % 100 == 0:
            print('.', end="")
            sys.stdout.flush()

    return NDCG / valid_user, HT / valid_user



## Model Definition (Inlined from model.py)


In [ ]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0


import numpy as np
import torch
import torch.nn as nn

class PointWiseFeedForward(torch.nn.Module):
    def __init__(self, hidden_units, dropout_rate):

        super(PointWiseFeedForward, self).__init__()

        self.conv1 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = torch.nn.Dropout(p=dropout_rate)
        self.relu = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = torch.nn.Dropout(p=dropout_rate)

    def forward(self, inputs):
        outputs = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(inputs.transpose(-1, -2))))))
        outputs = outputs.transpose(-1, -2) # as Conv1D requires (N, C, Length)
        outputs += inputs
        return outputs

# pls use the following self-made multihead attention layer
# in case your pytorch version is below 1.16 or for other reasons
# https://github.com/pmixer/TiSASRec.pytorch/blob/master/model.py

class SASRec(torch.nn.Module):
    def __init__(self, user_num, item_num, args):
        super(SASRec, self).__init__()

        self.user_num = user_num
        self.item_num = item_num
        self.dev = args.device
        self.hidden_units = args.hidden_units

        # TODO: loss += args.l2_emb for regularizing embedding vectors during training
        # https://stackoverflow.com/questions/42704283/adding-l1-l2-regularization-in-pytorch
        # self.item_emb = torch.nn.Embedding(self.item_num+1, args.hidden_units, padding_idx=0)
        self.item_emb = nn.Sequential(
            nn.Linear(args.item_emb_dim, 256),
            nn.LayerNorm(256),
            nn.ELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ELU(),
            nn.Linear(128, args.hidden_units),
        )

        self.pos_emb = torch.nn.Embedding(args.maxlen, args.hidden_units) # TO IMPROVE
        self.emb_dropout = torch.nn.Dropout(p=args.dropout_rate)

        self.attention_layernorms = torch.nn.ModuleList() # to be Q for self-attention
        self.attention_layers = torch.nn.ModuleList()
        self.forward_layernorms = torch.nn.ModuleList()
        self.forward_layers = torch.nn.ModuleList()

        self.last_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)

        for _ in range(args.num_blocks):
            new_attn_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.attention_layernorms.append(new_attn_layernorm)

            new_attn_layer =  torch.nn.MultiheadAttention(args.hidden_units,
                                                            args.num_heads,
                                                            args.dropout_rate)
            self.attention_layers.append(new_attn_layer)

            new_fwd_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.forward_layernorms.append(new_fwd_layernorm)

            new_fwd_layer = PointWiseFeedForward(args.hidden_units, args.dropout_rate)
            self.forward_layers.append(new_fwd_layer)

            # self.pos_sigmoid = torch.nn.Sigmoid()
            # self.neg_sigmoid = torch.nn.Sigmoid()

    def log2feats(self, log_seqs, seq_emb):
        seqs = self.item_emb(torch.from_numpy(seq_emb).float().to(self.dev))
        # --- FIX: Scale by sqrt(hidden_units) as per SASRec paper ---
        seqs *= self.hidden_units ** 0.5 
        positions = np.tile(np.array(range(log_seqs.shape[1])), [log_seqs.shape[0], 1]) # (#batch, #maxlen)
        
        seqs += self.pos_emb(torch.LongTensor(positions).to(self.dev))
        seqs = self.emb_dropout(seqs)

        timeline_mask = torch.BoolTensor(log_seqs == 0).to(self.dev)
        seqs *= ~timeline_mask.unsqueeze(-1) # broadcast in last dim

        tl = seqs.shape[1] # time dim len for enforce causality
        attention_mask = ~torch.tril(torch.ones((tl, tl), dtype=torch.bool, device=self.dev))

        for i in range(len(self.attention_layers)):
            seqs = torch.transpose(seqs, 0, 1)
            Q = self.attention_layernorms[i](seqs)
            mha_outputs, _ = self.attention_layers[i](Q, seqs, seqs, 
                                            attn_mask=attention_mask)

            seqs = Q + mha_outputs
            seqs = torch.transpose(seqs, 0, 1)

            seqs = self.forward_layernorms[i](seqs)
            seqs = self.forward_layers[i](seqs)
            seqs *=  ~timeline_mask.unsqueeze(-1)

        log_feats = self.last_layernorm(seqs) # (U, T, C) -> (U, -1, C)

        return log_feats

    def forward(self, log_seqs, seq_emb, pos_emb, neg_emb): # for training        
        # log_feats = self.log2feats(log_seqs) # user_ids hasn't been used yet
        log_feats = self.log2feats(log_seqs, seq_emb)
        seqs_out = self.item_emb(torch.from_numpy(seq_emb).float().to(self.dev))
        pos_embs = self.item_emb(torch.from_numpy(pos_emb).float().to(self.dev))
        neg_embs = self.item_emb(torch.from_numpy(neg_emb).float().to(self.dev))

        pos_logits = (log_feats * pos_embs).sum(dim=-1)
        neg_logits = (log_feats * neg_embs).sum(dim=-1)

        # pos_pred = self.pos_sigmoid(pos_logits)
        # neg_pred = self.neg_sigmoid(neg_logits)

        return pos_logits, neg_logits, seqs_out # pos_pred, neg_pred

    def predict(self, log_seqs, seq_emb, item_indices): # for inference
        log_feats = self.log2feats(log_seqs, seq_emb) # 1, maxlen, hidden_unit

        final_feat = log_feats[:, -1, :].float() # (1, hidden_unit)
        item_embs = self.item_emb(torch.from_numpy(item_indices).float().to(self.dev)) # 1, maxlen, hidden_unit
        logits = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1) # (1, maxlen, hidden_unit) x (1, hidden_unit, 1)

        # preds = self.pos_sigmoid(logits) # rank same item list for different users

        return logits # preds # (U, I)



## Preprocessing


In [ ]:
def preprocess():
    print(f'Loading {DATASET} dataset...')
    
    if LM_PRIOR in ('sbert', 'mpnet', 'bge'):
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(MODEL_ID_MAP[LM_PRIOR])
        # OPTIMIZATION: Maximize encoder throughput by using batch_size=256 and forcing GPU
        encode_fn = lambda texts: model.encode(texts, batch_size=256, device='cuda', show_progress_bar=True)
    elif LM_PRIOR == 'gemini':
        from google import genai
        import time
        
        # Use default client (no v1 override)
        client = genai.Client(api_key=GEMINI_API_KEY)
        
        def encode_fn(texts):
            results = []
            batch_size = 100  # Process 100 books at a time
            for i in tqdm(range(0, len(texts), batch_size), desc="Gemini API"):
                batch = texts[i:i+batch_size]
                success = False
                while not success:
                    try:
                        # Using the exact model string from your API key list
                        r = client.models.embed_content(model='gemini-embedding-2', contents=batch)
                        for emb in r.embeddings:
                            results.append(np.array(emb.values))
                        success = True
                        time.sleep(2) # 2-second delay between batches
                    except Exception as e:
                        print(f"\nRate limited or error. Sleeping 15s... ({e})")
                        time.sleep(15)
            return np.stack(results)

    if DATASET == 'bookcrossing':
        ratings_path = os.path.join(INPUT_DIR, 'BX-Book-Ratings.csv')
        books_path = os.path.join(INPUT_DIR, 'BX-Books.csv')
        
        if not os.path.exists(ratings_path):
            ratings_path = '/kaggle/input/bookcrossing-dataset/Book reviews/BX-Book-Ratings.csv'
            books_path = '/kaggle/input/bookcrossing-dataset/Book reviews/BX-Books.csv'
            
        print(f'Reading {ratings_path}')
        ratings = pd.read_csv(ratings_path, sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
        books = pd.read_csv(books_path, sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
        
        ratings.columns = [c.strip().strip('"') for c in ratings.columns]
        books.columns = [c.strip().strip('"') for c in books.columns]
        
        ratings = ratings[ratings['Book-Rating'].astype(str) != '0']
        user_counts = ratings['User-ID'].value_counts()
        item_counts = ratings['ISBN'].value_counts()
        
        valid_users = user_counts[user_counts >= 5].index
        valid_items = item_counts.index
        ratings = ratings[ratings['User-ID'].isin(valid_users) & ratings['ISBN'].isin(valid_items)]
        
        rate = ratings.rename(columns={'User-ID': 'user', 'ISBN': 'item'})
        user_col, item_col = 'user', 'item'
        
        book_meta = books[['ISBN', 'Book-Title', 'Book-Author']].drop_duplicates('ISBN')
        book_meta['text'] = book_meta['Book-Title'].fillna('') + ' ' + book_meta['Book-Author'].fillna('')
        book_meta_map = dict(zip(book_meta['ISBN'], book_meta['text']))
        
        unique_isbns = list(book_meta_map.keys())
        unique_texts = [book_meta_map[i] for i in unique_isbns]
        
        print('Encoding book metadata...')
        emb_matrix = encode_fn(unique_texts)
        meta_emb_map = {isbn: emb_matrix[i] for i, isbn in enumerate(unique_isbns)}
        
        def get_item_emb(orig_id):
            return meta_emb_map.get(orig_id, np.zeros(emb_matrix.shape[1]))
            
    elif DATASET == 'movielens':
        ratings_path = '/kaggle/input/movielens-25m-dataset/ml-25m/ratings.csv'
        movies_path = '/kaggle/input/movielens-25m-dataset/ml-25m/movies.csv'
        
        if not os.path.exists(ratings_path):
            ratings_path = '/kaggle/input/movielens/ratings.csv'
            movies_path = '/kaggle/input/movielens/movies.csv'
            
        print(f'Reading {ratings_path}')
        rate = pd.read_csv(ratings_path)
        movie = pd.read_csv(movies_path)
        
        user_col = 'userId'
        item_col = 'movieId'
        meta_col = 'genres'
        
        movie_meta_map = dict(zip(movie[item_col], movie[meta_col]))
        unique_meta = movie[meta_col].unique()
        print('Encoding genres...')
        emb_matrix = encode_fn(unique_meta)
        meta_emb_map = {m: emb_matrix[i] for i, m in enumerate(unique_meta)}
        
        def get_item_emb(orig_id):
            return meta_emb_map.get(movie_meta_map.get(orig_id, ''), np.zeros(emb_matrix.shape[1]))

    elif DATASET == 'amazon':
        data_path = '/content/drive/MyDrive/AmazonLM/Prime_Pantry_5.json.gz'
        if not os.path.exists(data_path):
            data_path = './Prime_Pantry_5.json.gz' # Fallback
            
        print(f'Reading {data_path}')
        rate = pd.read_json(data_path, lines=True, compression='gzip')
        user_col = 'reviewerID'
        item_col = 'asin'
        
        print(f"Extracting '{AMAZON_TEXT_COL}' for metadata...")
        rate[AMAZON_TEXT_COL] = rate[AMAZON_TEXT_COL].fillna('')
        item_meta = rate.groupby(item_col)[AMAZON_TEXT_COL].first().reset_index()
        
        movie_meta_map = dict(zip(item_meta[item_col], item_meta[AMAZON_TEXT_COL]))
        unique_meta = item_meta[AMAZON_TEXT_COL].unique()
        
        print('Encoding Amazon text...')
        emb_matrix = encode_fn(unique_meta)
        meta_emb_map = {m: emb_matrix[i] for i, m in enumerate(unique_meta)}
        
        def get_item_emb(orig_id):
            return meta_emb_map.get(movie_meta_map.get(orig_id, ''), np.zeros(emb_matrix.shape[1]))

    print('Mapping IDs to integers...')
    unique_users = rate[user_col].unique()
    unique_items = rate[item_col].unique()
    user_to_idx = {u: i+1 for i, u in enumerate(unique_users)}
    item_to_idx = {it: i+1 for i, it in enumerate(unique_items)}
    
    rate['user_idx'] = rate[user_col].map(user_to_idx)
    rate['item_idx'] = rate[item_col].map(item_to_idx)

    user_item, items = {}, {}
    for user_idx, grp in tqdm(rate.groupby('user_idx'), desc='Processing users'):
        user_item[user_idx] = grp['item_idx'].tolist()
        items[user_idx] = [get_item_emb(orig_id) for orig_id in grp[item_col]]

    with open('user_item_all.pkl', 'wb') as f: pickle.dump(user_item, f)
    with open('item_emb.pkl', 'wb') as f: pickle.dump(items, f)
    with open('id_mappings.pkl', 'wb') as f:
        pickle.dump({'user_to_idx': user_to_idx, 'item_to_idx': item_to_idx}, f)

    # Calculate Cold-Start items (items with < 5 interactions in TRAINING set)
    from collections import Counter
    train_users_limit = int(len(unique_users) * 0.8)
    train_interactions = []
    for u in range(1, train_users_limit + 1):
        if u in user_item:
            train_interactions.extend(user_item[u])
    
    counts = Counter(train_interactions)
    cold_items = {i_idx for i_idx, count in counts.items() if count < 5}
    with open('cold_items.pkl', 'wb') as f: pickle.dump(cold_items, f)

    print(f'Done. Users: {len(unique_users)}, Items: {len(unique_items)}, Cold Items: {len(cold_items)}')

preprocess()



## Distance Calculation


In [ ]:
def build_item_dist(k=K):
    with open('user_item_all.pkl', 'rb') as f: user_item = pickle.load(f)
    with open('item_emb.pkl', 'rb') as f: item_emb = pickle.load(f)

    print('Building unique item map...')
    item_unique_map = {}
    for user, item_ids in user_item.items():
        for i_id, emb in zip(item_ids, item_emb[user]):
            if i_id not in item_unique_map:
                item_unique_map[i_id] = emb

    unique_item_ids = sorted(item_unique_map.keys())
    max_id = max(unique_item_ids)
    emb_dim = item_unique_map[unique_item_ids[0]].shape[0]

    item_llm_matrix = np.zeros((max_id + 1, emb_dim))
    for i_id in unique_item_ids:
        item_llm_matrix[i_id] = item_unique_map[i_id]
    item_llm = [item_llm_matrix[i] for i in range(max_id + 1)]

    # OPTIMIZATION: PyTorch GPU Distance Calculation
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    item_llm_tensor = torch.from_numpy(item_llm_matrix).float().to(device)

    print(f'Computing KNN for {len(unique_item_ids)} items on {device} (k={k})...')
    item_dist = {}
    batch_size = 2000 # Much larger batch size for GPU
    
    with torch.no_grad():
        for i in tqdm(range(1, max_id + 1, batch_size), desc='KNN Batch'):
            end = min(i + batch_size, max_id + 1)
            batch = item_llm_tensor[i:end]
            dists = torch.cdist(batch, item_llm_tensor)
            _, indices = torch.topk(dists, k=k, largest=False)
            indices = indices.cpu().numpy()
            for j, idx_array in enumerate(indices):
                item_dist[i + j] = idx_array.tolist()

    with open('item_llm.pkl', 'wb') as f: pickle.dump(item_llm, f)
    with open('item_dist.pkl', 'wb') as f: pickle.dump(item_dist, f)
    print('Saved item_llm.pkl and item_dist.pkl')

build_item_dist()



## Training


In [ ]:
def train():
    with open('user_item_all.pkl', 'rb') as f: user_item = pickle.load(f)
    with open('item_emb.pkl', 'rb') as f: item_emb = pickle.load(f)
    with open('item_llm.pkl', 'rb') as f: item_llm = pickle.load(f)
    with open('item_dist.pkl', 'rb') as f: item_dist = pickle.load(f)
    item_llm = np.stack(item_llm)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    class Args:
        usernum = len(user_item)
        itemnum = max(item_dist.keys())
        item_emb_dim = item_emb[next(iter(item_emb))][0].shape[0]
        maxlen = 100
        train = True
        train_rate = 0.8
        l2_emb = 0
        num_blocks = 2
        num_heads = 4
        dropout_rate = 0.5
        hidden_units = HIDDEN_UNITS
        batch_size = BATCH_SIZE
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    args = Args()

    # OPTIMIZATION: Vectorized GPU Prior Calculation
    print('Pre-calculating LM prior weights (Vectorized on GPU)...')
    pre_seq_x = {}
    item_llm_t = torch.from_numpy(item_llm).float().to(device)
    
    batch_size_prior = 1000
    all_items = list(item_dist.keys())
    
    with torch.no_grad():
        for i in tqdm(range(0, len(all_items), batch_size_prior), desc='Prior Weights'):
            batch_items = all_items[i:i+batch_size_prior]
            
            # shape: (batch_size, K)
            neighbors_idx = torch.tensor([item_dist[it][:K] for it in batch_items], device=device)
            # shape: (batch_size, K, emb_dim)
            neighbors_llm = item_llm_t[neighbors_idx]
            
            # std shape: (batch_size, emb_dim)
            std = torch.std(neighbors_llm, dim=1)
            cov = 1 / (std + 1e-3)
            
            # center shape: (batch_size, 1, emb_dim)
            center_llm = item_llm_t[batch_items].unsqueeze(1)
            
            # dist_sq shape: (batch_size, K, emb_dim)
            dist_sq = (center_llm - neighbors_llm) ** 2
            
            # weighted shape: (batch_size, K)
            weighted = torch.sum(dist_sq * cov.unsqueeze(1), dim=-1)
            weights = torch.exp(-0.5 * weighted)[:, 1:] # Skip self
            
            for idx, item in enumerate(batch_items):
                pre_seq_x[item] = weights[idx]

    num_batch = len(user_item) // args.batch_size
    sampler = WarpSampler(user_item, item_emb, item_llm, args.usernum, args.itemnum,
                          args.item_emb_dim, args.train, args.train_rate,
                          batch_size=args.batch_size, maxlen=args.maxlen, n_workers=3)

    model = SASRec(args.usernum, args.itemnum, args).to(device).float()
    for _, p in model.named_parameters():
        try: torch.nn.init.xavier_normal_(p.data)
        except: pass

    bce = torch.nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98))

    t0, loss_best = time.time(), 1e10
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        loss_rec = 0
        for step in range(num_batch):
            _, seq, pos, seq_emb, pos_emb, neg_emb = sampler.next_batch()
            seq, pos = np.array(seq), np.array(pos)
            seq_emb, pos_emb, neg_emb = np.array(seq_emb), np.array(pos_emb), np.array(neg_emb)

            pos_logits, neg_logits, seqs_out = model(seq, seq_emb, pos_emb, neg_emb)
            pos_labels = torch.ones(pos_logits.shape, device=device)
            neg_labels = torch.zeros(neg_logits.shape, device=device)

            llm_emb_z = model.item_emb(torch.from_numpy(item_llm).float().to(device))
            seq_flat = seq.flatten()
            idx = np.where(seq_flat != 0)[0]
            if len(idx) == 0: continue
            seq_active = seq_flat[idx]
            seqs_out_active = seqs_out.reshape(-1, args.hidden_units)[idx, :].squeeze()

            neighbor_indices = [item_dist[it][:K] for it in seq_active]
            seq_z_neighbors = llm_emb_z[neighbor_indices]
            dist_sq_z = torch.sum((seqs_out_active.unsqueeze(1) - seq_z_neighbors)**2, dim=-1)[:, 1:]
            x_weights = torch.stack([pre_seq_x[it] for it in seq_active])
            loss_reg = torch.sum(x_weights * dist_sq_z)

            opt.zero_grad()
            idxs = np.where(pos != 0)
            loss = bce(pos_logits[idxs], pos_labels[idxs]) + bce(neg_logits[idxs], neg_labels[idxs]) + RHO * loss_reg
            loss_rec += loss.item()
            loss.backward()
            opt.step()

            if step % 50 == 0:
                print(f'Epoch {epoch} Step {step}/{num_batch} | loss={loss.item():.4f}')

        avg = loss_rec / num_batch
        print(f'--- Epoch {epoch} done | avg_loss={avg:.4f} ---')
        if avg < loss_best:
            loss_best = avg
            torch.save(model.state_dict(), f'./model_save/{RUN_NAME}_best.pth')
            print(f'  Saved best model.')

    print(f'Training complete in {time.time()-t0:.1f}s')
    sampler.close()

train()



## Evaluate


In [ ]:
def evaluate():
    with open('user_item_all.pkl', 'rb') as f: user_item = pickle.load(f)
    with open('item_emb.pkl', 'rb') as f: item_emb = pickle.load(f)
    with open('item_llm.pkl', 'rb') as f: item_llm = pickle.load(f)
    with open('item_dist.pkl', 'rb') as f: item_dist = pickle.load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class Args:
        usernum = len(user_item)
        itemnum = max(item_dist.keys())
        item_emb_dim = item_emb[next(iter(item_emb))][0].shape[0]
        maxlen = 100
        num_blocks = 2
        num_heads = 4
        dropout_rate = 0.5
        hidden_units = HIDDEN_UNITS
        train_rate = 0.8
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    args = Args()

    model = SASRec(args.usernum, args.itemnum, args).to(device).float()
    model_path = f'./model_save/{RUN_NAME}_best.pth'
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f'Loaded: {model_path}')

    item_llm_array = np.stack(item_llm)
    with open('cold_items.pkl', 'rb') as f: cold_items = pickle.load(f)
    
    NDCG_10, HT_10, NDCG_20, HT_20, valid_user = 0.0, 0.0, 0.0, 0.0, 0.0
    cs_NDCG_10, cs_HT_10, cs_NDCG_20, cs_HT_20, cs_valid_user = 0.0, 0.0, 0.0, 0.0, 0.0
    
    # Filter valid test users
    test_users = [u for u in range(int(args.usernum * args.train_rate) + 1, args.usernum + 1) if u in user_item and len(user_item[u]) >= 2]
    
    # OPTIMIZATION: Batched GPU Evaluation
    eval_batch_size = 128
    
    for i in tqdm(range(0, len(test_users), eval_batch_size), desc='Evaluating (Batched)'):
        batch_users = test_users[i:i+eval_batch_size]
        
        seqs = []
        seq_embs = []
        target_ids = []
        
        for user in batch_users:
            target_ids.append(user_item[user][-1])
            seq = np.zeros([args.maxlen], dtype=np.int32)
            idx = args.maxlen - 1
            for it in reversed(user_item[user][:-1]):
                seq[idx] = it; idx -= 1
                if idx == -1: break
            seqs.append(seq)
            
            seq_emb = [np.zeros(args.item_emb_dim)] * args.maxlen
            idx = args.maxlen - 1
            for e in reversed(item_emb[user][:-1]):
                seq_emb[idx] = e; idx -= 1
                if idx == -1: break
            seq_embs.append(np.vstack(seq_emb))
            
        seqs = np.array(seqs)
        seq_embs = np.array(seq_embs)
        
        with torch.no_grad():
            logits = model.predict(seqs, seq_embs, item_llm_array)
            logits = logits.cpu().numpy()
            
        for b, user in enumerate(batch_users):
            target_id = target_ids[b]
            rank = (-logits[b]).argsort().argsort()[target_id]
            is_cold = target_id in cold_items
            
            valid_user += 1
            if rank < 10: NDCG_10 += 1/np.log2(rank+2); HT_10 += 1
            if rank < 20: NDCG_20 += 1/np.log2(rank+2); HT_20 += 1
            
            if is_cold:
                cs_valid_user += 1
                if rank < 10: cs_NDCG_10 += 1/np.log2(rank+2); cs_HT_10 += 1
                if rank < 20: cs_NDCG_20 += 1/np.log2(rank+2); cs_HT_20 += 1

    results = {
        'run': RUN_NAME,
        'Overall_NDCG@10': round(NDCG_10/valid_user, 4),
        'Overall_HT@10':   round(HT_10/valid_user, 4),
        'CS_NDCG@10':      round(cs_NDCG_10/cs_valid_user, 4) if cs_valid_user > 0 else 0,
        'CS_HT@10':        round(cs_HT_10/cs_valid_user, 4) if cs_valid_user > 0 else 0,
        'CS_Count':        int(cs_valid_user)
    }
    print('\n--- Evaluation Results ---')
    for k, v in results.items(): print(f'{k}: {v}')

    import json
    results_file = './results/all_results.json'
    all_results = []
    if os.path.exists(results_file):
        with open(results_file) as f: all_results = json.load(f)
    all_results.append(results)
    with open(results_file, 'w') as f: json.dump(all_results, f, indent=2)
    return results

results = evaluate()

